# 📝 MarkItDown Web UI (Google Colab)

Notebook này giúp bạn chạy giao diện Web cho **MarkItDown** (công cụ chuyển đổi tài liệu sang Markdown của Microsoft) trực tiếp trên Google Colab và chia sẻ ra Internet thông qua đường link public của **Gradio**.

### 🚀 Hướng dẫn nhanh:
1. Nhấn nút **Run** (▶) ở **Bước 1: Cài đặt thư viện**.
2. Nhấn nút **Run** (▶) ở **Bước 2: Khởi chạy Web UI**.
3. Mở đường link **Public URL** (dạng `https://xxxx.gradio.live`) được tạo ra bên dưới ô kết quả để truy cập giao diện từ bất kỳ thiết bị nào!

---
### 📄 Các định dạng hỗ trợ:
- **Văn phòng**: PDF, Word (`.docx`), PowerPoint (`.pptx`), Excel (`.xlsx`)
- **Dữ liệu & Web**: CSV, JSON, XML, HTML
- **Sách & Lưu trữ**: EPUB, ZIP, TXT

### 📦 Bước 1: Cài đặt thư viện cần thiết (`markitdown` & `gradio`)

In [ ]:
!pip install -q "markitdown[all]" gradio

### 🌐 Bước 2: Khởi chạy Web UI và tạo link Public ra Internet
> Sau khi chạy cell dưới đây, hãy tìm dòng `Running on public URL: https://xxxxxxxx.gradio.live`

In [ ]:
import os
import tempfile
import pathlib
import gradio as gr
from markitdown import MarkItDown

# Khởi tạo đối tượng MarkItDown
markitdown = MarkItDown()

def convert_document(uploaded_file):
    if uploaded_file is None:
        return (
            "⚠️ Vui lòng tải lên một tệp tài liệu trước khi bấm chuyển đổi.",
            "",
            None,
            "Chưa có tệp nào được chọn."
        )
    
    file_path = uploaded_file if isinstance(uploaded_file, str) else uploaded_file.name
    original_name = pathlib.Path(file_path).stem
    
    try:
        result = markitdown.convert(file_path)
        md_content = result.text_content
        
        if not md_content or not md_content.strip():
            md_content = "_Tệp tài liệu này không chứa nội dung văn bản hoặc nội dung rỗng._"

        temp_dir = tempfile.mkdtemp()
        output_filename = f"{original_name}.md"
        output_path = os.path.join(temp_dir, output_filename)
        
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(result.text_content)
            
        status_msg = f"✅ Chuyển đổi thành công '{pathlib.Path(file_path).name}' ({len(result.text_content):,} ký tự)"
        
        return (
            md_content,
            result.text_content,
            output_path,
            status_msg
        )
    except Exception as e:
        error_msg = f"❌ Lỗi khi chuyển đổi: {str(e)}"
        return (
            f"**Đã xảy ra lỗi:**\n```\n{str(e)}\n```",
            "",
            None,
            error_msg
        )

def clear_all():
    return None, "", "", None, "Đã làm mới giao diện."

custom_css = """
.main-title { text-align: center; margin-bottom: 0.5rem; }
.sub-title { text-align: center; color: #666; margin-bottom: 1.5rem; }
.output-box { min-height: 400px; }
"""

with gr.Blocks(title="MarkItDown Web UI", css=custom_css, theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 📝 MarkItDown Web Converter
        ### Chuyển đổi mọi tài liệu (PDF, Word, Excel, PowerPoint, Text, HTML...) sang Markdown
        """,
        elem_classes=["main-title", "sub-title"]
    )
    
    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(
                label="📁 Tải lên tệp tài liệu",
                file_types=[
                    ".pdf", ".docx", ".doc", ".pptx", ".ppt",
                    ".xlsx", ".xls", ".csv", ".json", ".xml",
                    ".html", ".htm", ".txt", ".epub", ".zip"
                ],
                type="filepath"
            )
            
            with gr.Row():
                btn_convert = gr.Button("⚡ Chuyển đổi ngay", variant="primary")
                btn_clear = gr.Button("🔄 Làm mới", variant="secondary")
                
            status_output = gr.Textbox(
                label="Trạng thái",
                value="Sẵn sàng tiếp nhận tài liệu.",
                interactive=False
            )
            
            download_output = gr.File(
                label="📥 Tải xuống tệp .md",
                interactive=False
            )
            
            gr.Markdown(
                """
                > **Định dạng hỗ trợ:**
                > - **Văn phòng:** PDF, Word (.docx), PowerPoint (.pptx), Excel (.xlsx)
                > - **Dữ liệu & Web:** CSV, JSON, XML, HTML
                > - **Sách & Lưu trữ:** EPUB, ZIP, TXT
                """
            )

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem("👁️ Xem trước (Rendered Markdown)"):
                    md_preview = gr.Markdown(
                        value="_Chưa có nội dung. Vui lòng tải file và bấm chuyển đổi._",
                        elem_classes=["output-box"]
                    )
                with gr.TabItem("💻 Mã nguồn Markdown (Raw Text)"):
                    raw_preview = gr.Textbox(
                        label="Mã Markdown thô",
                        placeholder="Nội dung Markdown sẽ xuất hiện tại đây...",
                        lines=18,
                        show_copy_button=True,
                        interactive=False
                    )

    btn_convert.click(
        fn=convert_document,
        inputs=[file_input],
        outputs=[md_preview, raw_preview, download_output, status_output]
    )
    
    file_input.upload(
        fn=convert_document,
        inputs=[file_input],
        outputs=[md_preview, raw_preview, download_output, status_output]
    )
    
    btn_clear.click(
        fn=clear_all,
        inputs=[],
        outputs=[file_input, md_preview, raw_preview, download_output, status_output]
    )

# Khởi chạy trên Google Colab với share=True để lấy public link
demo.launch(share=True, debug=True)
